# Loka Research Agent - LangGraph Version

Build a research agent about **Loka** step by step with
**[LangGraph](https://langchain-ai.github.io/langgraph/)**.

This notebook is a **guided hands-on exercise** that you will follow throughout the workshop. The agent will be built progressively in three exercises, each adding more capabilities. You will find the following markers in the notebook to guide you:

| Marker               | Meaning                                              |
|----------------------|------------------------------------------------------|
| 🛠️ **Setup**        | Run this once to set up the environment and imports. |
| ✅ **Given**          | Code or instructions provided for you                |
| ✏️ **Your turn**     | Code you need to write to complete the exercise.     |
| 🚀 **Going further** | Optional stretch ideas if you finish early.          |
| 📚 **Hints**         | Links into the LangGraph docs for more information.  |

LangGraph is more explicit than Strands: you write the graph that wires the pieces together instead of handing a model a list of tools. That means more code overall -- so in this notebook **more of it is already given** (state, node functions, the run loop). Your turns are the handful of lines that actually make the difference: adding a tool, binding it to the model, wiring an edge, or turning on memory. The graph shape barely changes between exercises, so once you've wired it in Exercise 1, it's given as already-done from Exercise 2 onward.

The exercises contain detailed instructions on what to do, but you are encouraged to explore and experiment. Refer to the LangGraph documentation for more information on the concepts and APIs used in this notebook.

## 🛠️ Setup

Run this cell once to set up the environment and imports. Make sure you have already configured an `.env` file with your **Anthropic API key** and run `uv sync` to have the project dependencies installed. If you haven't done this yet, follow the instructions in the `README.md` file.

In [1]:
import os, sys
from pathlib import Path

# Find the repo root
ROOT = next(b for b in (Path.cwd(), *Path.cwd().parents) if (b / "shared").is_dir())
sys.path.insert(0, str(ROOT / "shared"))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")
assert os.getenv("ANTHROPIC_API_KEY"), "Add ANTHROPIC_API_KEY to your .env (copy .env.example)."

from langchain_anthropic import ChatAnthropic
from knowledge_base import search_documents, list_topics
from website import search_website

model = ChatAnthropic(
    model=os.getenv("ANTHROPIC_MODEL", "claude-haiku-4-5-20251001"),
    api_key=os.environ["ANTHROPIC_API_KEY"],
    max_tokens=1024,
    temperature=0.3,
)

print("Model ready:", model.model)

Model ready: claude-haiku-4-5-20251001


## ✅ The knowledge base (given)

Your agent's knowledge lives in `shared/`, already written for you:
`search_documents(query)`, `list_topics()`, and `search_website(query)`. In the
exercises you'll wrap these as **tools**. Run this to see what they return (no API
key needed):

In [2]:
print(list_topics())
print("\n--- search_documents('learning') ---\n")
print(search_documents("learning"))

The Loka knowledge base covers these topics:
- AWS Innovation Partner of the Year
- Time Off and the 5/4 Friday Schedule
- Fully Remote, Work From Anywhere
- In-Person Connection Despite Being Remote
- Cutting-Edge Client Projects
- Multicultural, Global Team
- Learning and Development
- Culture of Innovation and Internal Initiatives

--- search_documents('learning') ---

## Learning and Development
One of Loka's headline goals for 2026 is to become one of the world's best learning organizations. That means real budget and real time for growth: courses, certifications, and the expectation that you keep leveling up. Learning is treated as part of the job, not a side quest.


In [3]:
print("--- search_website('services') ---\n")
print(search_website("what customers does Loka work with?"))

--- search_website('services') ---

(source: live)

[https://www.loka.com/about]
AWS, Jeff oversees Global Direct Sales at Loka, helping SMB customers unlock value through agentic AI, modernization and cloud migration. A culture of innovation Our team members live by the credo What you develop matters. We work wherever we’re most productive and take every other

[https://www.loka.com/]
17 What Fascinates Sol Rashidi? Navigating AI, balancing the possible with the practical and the art of going rogue. Jump to episode Jump to episode Blog Life at Loka • 7.8.26 Loka's Explorer Program: The Best Decision I Almost Didn't Make What an ML Engineer

[https://www.loka.com/]
Cameo. Their deep GenAI, ML and AWS expertise, paired with industry insight and exceptional customer care, makes them a standout among consultancies." Dom Scandinaro CTO , Cameo “In my 35 years of work experience, I have not come across a consulting partner like Loka. They


## Exercise 1: Basic Agent

An agent in LangGraph is a **graph**: a small state machine you wire together
yourself. The state holds the conversation (`messages`), one **node** calls the
model, another **node** runs tools, and a **conditional edge** decides whether to
loop back to the model or stop.

That wiring is the piece Strands' `Agent` class did for you automatically. You'll
build it once, here, in this exercise -- its shape barely changes for the rest of
the notebook, so from Exercise 2 onward it'll already be wired for you.

Your job: turn the given `search_documents` into a tool (same idea as the Strands
version), bind it to the model, and connect the graph: a node that calls the
model, a node that runs the tool, and the router between them.

> 💡 The tool's **docstring** is what the model reads to decide when to use it --
> write it for the model, same as in the Strands notebook.

**📚 Hints**
- [`tools_condition` prebuilt router](https://langchain-ai.github.io/langgraph/reference/prebuilt/#langgraph.prebuilt.tool_node.tools_condition)
- [`ToolNode` prebuilt](https://langchain-ai.github.io/langgraph/reference/prebuilt/#langgraph.prebuilt.tool_node.ToolNode)

### ✅ Given: the state and the system prompt

The **state** is what flows through every node in the graph -- here, just the
running list of messages. `add_messages` tells LangGraph to *append* new
messages to it instead of you managing the list by hand (this is the "State =
Memory" building block the rest of the notebook builds on).

In [4]:
from typing import Annotated, TypedDict

from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition


class State(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]


SYSTEM_PROMPT = """You are the Loka Research Agent, a friendly assistant that \
answers questions about Loka (the company).

- Always answer from the knowledge base via search_knowledge_base; do not rely \
on prior knowledge about Loka.
- If the knowledge base has no answer, say so instead of guessing.
- Be concise, warm, and a little proud of how great Loka is to work at.
"""

### ✏️ Your turn -- TODO 1: make a tool

Wrap the given `search_documents(query)` as a LangChain `@tool` and add a
docstring for the model to read (same idea as the Strands version -- the
docstring is the model-facing contract).

In [5]:
@tool
def search_knowledge_base(query: str) -> str:
    """Search Loka's internal knowledge base for information about the company,
    its benefits, culture, and how it works.

    Args:
        query: A short natural-language description of what to look for.
    """
    return search_documents(query)

### ✏️ Your turn -- TODO 2: bind the tool to the model

`.bind_tools([...])` is what lets the model *request* a tool call in its
output -- the `ToolNode` you'll wire next is what actually runs it.

In [6]:
model_with_tools = model.bind_tools([search_knowledge_base])

### ✅ Given: the node that calls the model

This node just hands the running state to the model and appends its reply.
It's short and won't change for the rest of the notebook.

In [7]:
def agent_node(state: State) -> dict:
    return {"messages": [model_with_tools.invoke(state["messages"])]}

### ✏️ Your turn -- TODO 3: wire the graph

Two nodes are registered for you below (`agent_node`, and a `ToolNode` running
your tool). Connect them:
- an edge from `START` into `"agent"`
- a **conditional edge** out of `"agent"`, using `tools_condition` as the
  router -- it reads the model's last message and returns `"tools"` if it asked
  for one, otherwise it should end the run
- an edge from `"tools"` back to `"agent"`, so the loop continues after the
  tool runs

In [8]:
builder = StateGraph(State)
builder.add_node("agent", agent_node)
builder.add_node("tools", ToolNode([search_knowledge_base]))

builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition, {"tools": "tools", END: END})
builder.add_edge("tools", "agent")

graph = builder.compile()

### ✅ Given: run it

In [9]:
result = graph.invoke({
    "messages": [SystemMessage(SYSTEM_PROMPT), HumanMessage("What is Loka's time-off policy?")]
})

for message in result["messages"]:
    if getattr(message, "tool_calls", None):
        for call in message.tool_calls:
            print(f"[tool call] {call['name']}({call['args']})")
    elif message.type == "tool":
        print(f"[tool result] {message.content}\n")
print(result["messages"][-1].content)

[tool call] search_knowledge_base({'query': 'time-off policy vacation days paid time off'})
[tool result] ## Time Off and the 5/4 Friday Schedule
Loka runs a 5/4 schedule: every other Friday is off. That is 26 extra days off per year on top of regular vacation. Roughly one long weekend every two weeks, permanently, forever. Work-life balance is not a perk here, it is the calendar.

## Multicultural, Global Team
Loka is genuinely multicultural: teammates in Portugal, Brazil, Macedonia, the USA, and beyond. Daily standups double as a small tour of world time zones and coffee habits.

## Learning and Development
One of Loka's headline goals for 2026 is to become one of the world's best learning organizations. That means real budget and real time for growth: courses, certifications, and the expectation that you keep leveling up. Learning is treated as part of the job, not a side quest.

Great question! Here's what makes Loka's time-off policy pretty special:

**The 5/4 Friday Schedule**: L

### 🎯 Ask your own

In [11]:
result = graph.invoke({
    "messages": [
        SystemMessage(SYSTEM_PROMPT),
        HumanMessage("What kind of projects does Loka's engineering team typically work on?"),
    ]
})
print(result["messages"][-1].content)

Great question! Based on what I found, Loka's engineering team works on **cutting-edge projects** that leverage the newest AI tools and frameworks. Here's what that means:

- **Generative AI projects** – building with the latest in AI technology
- **Agent frameworks** – working with modern agent-based architectures
- **Modern cloud architectures** – deploying on contemporary cloud infrastructure

The exciting part? You're not maintaining legacy systems or old codebases. Instead, you're shipping with the tools and technologies that are being discussed at major tech conferences *right now*. It's the kind of work that keeps you on the cutting edge of the industry.

Plus, you get to do all this from anywhere in the world as part of a genuinely multicultural team! 🌍

Is there anything else you'd like to know about working at Loka?


### 🚀 Going further

If you finished early or want to explore more, try these ideas:

- **Inspect the run.** Print `result["messages"]` in full to see how LangGraph
  represents the conversation as a list of typed messages (system/human/ai/tool).
- **Use a prebuilt LangChain tool** instead of a custom one, e.g. from
  `langchain_community.tools`. Add it to both the `ToolNode` and the
  `bind_tools([...])` list.
- **Change the persona** in `SYSTEM_PROMPT` (formal? pirate?) and re-run.
- **Ask something NOT in the knowledge base** and see if the agent admits it doesn't know.
- **Visualize the graph** with `graph.get_graph().print_ascii()`.